# PORTFOLIO SIMULATION

We simulate the value of three portfolios with arbitrary weights.
For each group of assets, we simulate one portfolio in which the assets are independent and another in which they are correlated using the Cholesky method.

In [ ]:
w1= [1,0,0,0]
w2= [0.25,0.25,0.25,0.25]
w3= [0.1,0.2,0.3,0.4]
w= [w1,w2,w3]

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

In [ ]:
prices=yf.download(["NVDA", "BIMBOA.MX", "CEMEXCPO.MX", "AZN"], start="2023-01-01").Close.dropna()

In [ ]:
V0=100_000
returns=np.log(prices/prices.shift(1)).dropna()
r=np.mean(returns, axis=0)*252
std=np.std(returns, axis=0)*np.sqrt(252)
s0=prices.iloc[-1]
# NVDA is quoted in USD, and I need it in pesos; the same applies to AZN, but we convert them at the end.

In [ ]:
def create_price(S0, r, sigma, T, n_paths):
    z = np.random.normal(0, 1, n_paths)
    price = S0 * np.exp(
         (r - 0.5 * sigma**2) * T
         + sigma * np.sqrt(T) * z
    )
    return price

In [ ]:
corr_matrix=returns.corr()
epsilon=np.random.normal(0,1,[4,10_000])
L=np.linalg.cholesky(corr_matrix)
z=L@epsilon
T=1

In [ ]:
def st_cholesky(s0,mu,std,T,n_paths, corr_matrix):
    i=0
    n_assets=len(list(mu))
    epsilon=np.random.normal(0,1,[n_assets,n_paths])
    L=np.linalg.cholesky(corr_matrix)
    z=L@epsilon
    st = {}
    for ticker in s0.keys() :
        print(ticker)
        st[ticker] = s0.loc[ticker]*np.exp((mu.loc[ticker] - std.loc[ticker]**2/2)*T + std.loc[ticker]*np.sqrt(T)*z[i, :])

        i += 1
    return st
assets_corr=st_cholesky(s0,r,std,1,10_000,corr_matrix)

We omit the mu term because we want Z to follow a normal distribution with mean 0.

z is a matrix of size (# assets, m simulations).

In [ ]:
assets_iid={}
for index in r.index:
    price=create_price(s0[index],r[index],std[index],1,10_000)
    assets_iid[index] = price


In [ ]:
usd_mxn_prices=yf.download(["USDMXN=X"], start="2023-01-01").Close.dropna()
usd_mxn_prices


In [ ]:
fx_returns=np.log(usd_mxn_prices/usd_mxn_prices.shift(1))
fx_mean_return=np.mean(fx_returns)*252
fx_std=np.std(fx_returns)*np.sqrt(252)
fx_spot=usd_mxn_prices.iloc[-1,0]

In [ ]:
usd_mxn = create_price(fx_spot, fx_mean_return, fx_std, 1, 10_000)

In [ ]:
assets_iid["AZN"]=assets_iid["AZN"]*usd_mxn
assets_iid["NVDA"]=assets_iid["NVDA"]*usd_mxn
assets_corr["AZN"]=assets_corr["AZN"]*usd_mxn
assets_corr["NVDA"]=assets_corr["NVDA"]*usd_mxn


In [ ]:
s0["NVDA"] *= fx_spot
s0["AZN"] *= fx_spot

In [ ]:
portfolio_units=np.array(w)*V0/np.array(list(s0.values))
Vt_iid=portfolio_units@np.array(list(assets_iid.values()))
Vt_corr=portfolio_units@np.array(list(assets_corr.values()))

In [ ]:
type(assets_iid.values())

In [ ]:
for i in range(3):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))

    ax[0].hist(Vt_iid[i], bins=30)
    ax[0].set_title(f"Independent portfolio with weights {w[i]}")

    ax[1].hist(Vt_corr[i], bins=30)
    ax[1].set_title(f"Correlated portfolio with weights {w[i]}")

    plt.tight_layout()
    plt.show()

## Comparison metrics

In [ ]:
for i in range(3):
    metrics = pd.DataFrame({
        "Mean": [np.mean(Vt_iid[i]), np.mean(Vt_corr[i])],
        "Std. deviation": [np.std(Vt_iid[i]), np.std(Vt_corr[i])],
        "VaR 95%": [V0-np.percentile(Vt_iid[i], 5), V0-np.percentile(Vt_corr[i], 5)],
        "Kurtosis": [pd.Series(Vt_iid[i]).kurt(), pd.Series(Vt_corr[i]).kurt()]
    }, index=["Independent", "Correlated"])
    print(f"Portfolio {i+1} with weights {w[i]}")
    display(metrics)

### Conclusions

The mean is similar in both types of simulation because correlation mainly affects risk rather than expected return. There are almost no changes in the first portfolio because it contains only one asset.

For portfolios 2 and 3, the standard deviation and VaR make it possible to compare the level of risk. If they are higher in the correlated portfolio, this means that correlation reduces the benefit of diversification. Higher kurtosis indicates that the distribution has heavier tails and a greater presence of extreme values.

# EXERCISE 2

In [ ]:
prices=yf.download(["NVDA", "AAPL", "MSFT"], start="2023-01-01").Close.dropna()

In [ ]:
returns=np.log(prices/prices.shift(1)).dropna()

In [ ]:
R=returns.corr()

In [ ]:
s0=prices.iloc[-1]
r=np.mean(returns, axis=0)*252
std=np.std(returns, axis=0)*np.sqrt(252)
np.transpose(z)

In [ ]:
s0.keys()

In [ ]:
assets_corr=st_cholesky(s0,r,std,1,10_000,R)

In [ ]:
w1= [1,0,0]
w2= [0.33,0.33,0.34]
w3= [0.2,0.2,0.6]
w= [w1,w2,w3]


In [ ]:
def create_price(S0, r, sigma, T, n_paths):
    z = np.random.normal(0, 1, n_paths)
    price = S0 * np.exp(
         (r - 0.5 * sigma**2) * T
         + sigma * np.sqrt(T) * z
    )

    return price

In [ ]:
assets_iid={}
for index in r.index:
    price=create_price(s0[index],r[index],std[index],1,10_000)
    assets_iid[index] = price


In [ ]:
portfolio_units=np.array(w)*V0/np.array(list(s0.values))
Vt_corr=portfolio_units@np.array(list(assets_corr.values()))
Vt_iid=portfolio_units@np.array(list(assets_iid.values()))

In [ ]:
for i in range(3):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))

    ax[0].hist(Vt_iid[i], bins=30)
    ax[0].set_title(f"Independent portfolio with weights {w[i]}")

    ax[1].hist(Vt_corr[i], bins=30)
    ax[1].set_title(f"Correlated portfolio with weights {w[i]}")

    plt.tight_layout()
    plt.show()

## Comparison metrics

In [ ]:
for i in range(3):
    metrics = pd.DataFrame({
        "Mean": [np.mean(Vt_iid[i]), np.mean(Vt_corr[i])],
        "Std. deviation": [np.std(Vt_iid[i]), np.std(Vt_corr[i])],
        "VaR 95%": [V0-np.percentile(Vt_iid[i], 5), V0-np.percentile(Vt_corr[i], 5)],
        "Kurtosis": [pd.Series(Vt_iid[i]).kurt(), pd.Series(Vt_corr[i]).kurt()]
    }, index=["Independent", "Correlated"])
    print(f"Portfolio {i+1} with weights {w[i]}")
    display(metrics)

### Conclusions

The mean is again similar between the two simulations. In portfolio 1, correlation has no effect because the entire investment is allocated to a single asset.

In portfolios 2 and 3, the correlated simulation may have a higher standard deviation and VaR because AAPL, MSFT, and NVDA tend to move in the same direction. Kurtosis helps identify which simulation has a greater presence of extreme values. Portfolio 2 is more diversified, while portfolio 3 depends more heavily on NVDA because of its 60% weight.